# B1+ KLUE-RoBERTa 실행과 H6 임베딩 (Colab L4)

**준비 (한 번만)**
1. 바탕화면 `AIX_klue` 폴더를 Google Drive `내 드라이브` 바로 아래에 올린다. 안에는 `data/`(H4 의 `meta_trainval.csv`, `split_v1_trainval.csv`, `ocr_trainval.csv`)만 있다. 이미지는 쓰지 않는다(OCR 글만 본다).
2. 코드는 비공개 저장소 `chlee1027/AI_X` 의 `image-spec` 브랜치에서 받는다. Colab 왼쪽 열쇠 아이콘(보안 비밀)에 이름 `GITHUB_TOKEN`, 값 = GitHub 토큰(저장소 읽기 권한)을 넣고 「노트북 액세스」를 켠다.
3. 런타임 > 런타임 유형 변경 > L4 GPU. 그다음 셀을 위에서부터 차례로 실행.

**하는 일:** 미세조정 8번 = 검증 점수용 시드 3회(0, 1, 2) + H6 학습 몫 임베딩용 group_id 5겹 OOF.
학습 몫 임베딩을 OOF 로 뽑는 것은 자기 학습 데이터를 본 모델의 임베딩으로 M2 를 학습하면 점수가 부풀기 때문이다.

**결과:** `내 드라이브/AIX_klue/runs/` 와 다영에게 넘길 `내 드라이브/AIX_klue/H6_다영.zip`. 결과 파일의 commit 칸에 받은 저장소 커밋이 찍힌다.

In [ ]:
# 1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Drive 연결, 저장소 받기, 데이터 복사 (Drive 에서 바로 돌리면 느리다)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, shutil, subprocess
SRC = '/content/drive/MyDrive/AIX_klue'
assert os.path.isdir(f'{SRC}/data'), f'{SRC}/data 가 없다. 바탕화면 AIX_klue 폴더를 내 드라이브 바로 아래에 올렸는지 확인'
tok = userdata.get('GITHUB_TOKEN')
if os.path.isdir('/content/AI_X'):
    shutil.rmtree('/content/AI_X')
r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', 'image-spec', f'https://{tok}@github.com/chlee1027/AI_X.git', '/content/AI_X'], capture_output=True, text=True)
assert r.returncode == 0, r.stderr.replace(tok, '***')
del tok
if os.path.isdir('/content/data'):
    shutil.rmtree('/content/data')
shutil.copytree(f'{SRC}/data', '/content/data')
os.chdir('/content/AI_X/image_models')
!git log --oneline -1
!ls -la /content/data

In [ ]:
# 3. 패키지 (README 에 고정한 판)
!pip -q install transformers==4.46.3
import transformers, torch, sklearn, pandas
print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 4. 데이터 점검: 평가(test) 몫이 없고, 학습·검증이 group·site 로 겹치지 않는지
import pandas as pd
m = pd.read_csv('/content/data/meta_trainval.csv', dtype=str, keep_default_na=False)
print(m['split'].value_counts().to_dict())
assert set(m['split']) <= {'train', 'val'}, '평가 몫이 들어 있다. 멈춘다'
for c in ['group_id', 'site']:
    both = (m.groupby(c)['split'].nunique() > 1).sum()
    print(c, '학습·검증 모두에 걸친 수:', both)
    assert both == 0
sc = m[m['unsure'] != '1']
print('채점 대상 확실한 불법(train/val):', sc[sc.label == 'illegal']['split'].value_counts().to_dict())

In [ ]:
# 5. 학습 + 검증 점수 + H6 임베딩 (미세조정 8번)
import time
t0 = time.time()
!python b1plus_klue.py --split /content/data/split_v1_trainval.csv --meta /content/data/meta_trainval.csv --ocr /content/data/ocr_trainval.csv --config configs/b1plus_klue.yaml --out runs
print(f'걸린 시간 {(time.time() - t0) / 60:.1f}분')

In [ ]:
# 6. 결과 요약
import json
rep = json.load(open('runs/b1plus_klue/report_b1plus_klue.json'))
for i, r in enumerate(rep['val_by_seed']):
    print('seed', i, {k: r[k] for k in ('precision', 'recall', 'f1', 'pr_auc', 'threshold') if k in r})
print('시드 평균 ± 표준편차:', rep['val_mean_std'])
import numpy as np
for s in ['val', 'train_oof']:
    e = np.load(f'runs/b1plus_klue/emb_b1plus_klue_{s}.npy'); ids = pd.read_csv(f'runs/b1plus_klue/emb_b1plus_klue_{s}_ids.csv')
    print(s, e.shape, e.dtype, '| ids', len(ids), '| 0 행', int((np.abs(e).sum(1) == 0).sum()))

In [ ]:
# 7. Drive 로 저장: runs 전체(체크포인트 포함) + H6 묶음
import zipfile, glob
dst = os.path.join(SRC, 'runs')
if os.path.isdir(dst):
    shutil.rmtree(dst)
shutil.copytree('runs', dst)
r = 'runs/b1plus_klue'
h6 = [f'{r}/emb_b1plus_klue_{s}{x}' for s in ('val', 'train_oof') for x in ('.npy', '_ids.csv')]
h6 += ['b1plus_klue.py', 'imgcommon.py', 'configs/b1plus_klue.yaml', f'{r}/report_b1plus_klue.json']
with zipfile.ZipFile(os.path.join(SRC, 'H6_다영.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    for f in h6:
        z.write(f, os.path.basename(f) if f.startswith('runs') else f)
print('저장:', dst)
print('H6 묶음:', os.path.join(SRC, 'H6_다영.zip'), f"{os.path.getsize(os.path.join(SRC, 'H6_다영.zip')) / 1e6:.1f}MB")
!ls -la {SRC}

## 끝나면

1. 셀 6 의 출력(시드별 점수, 평균 ± 표준편차, 임베딩 모양)을 Claude 에게 붙여 넣는다.
2. `내 드라이브/AIX_klue/H6_다영.zip` 을 다영에게 공유한다. 행 순서는 같은 이름의 `_ids.csv` 를 따른다(768차원 float16).
3. 이 노트북을 실행 결과가 남은 채로 내려받아(파일 > 다운로드 > .ipynb) 바탕화면에 두면 저장소로 옮긴다.
4. 체크포인트 `runs/b1plus_klue/ckpt_s0` 는 11.16 평가셋 추론 노트북에서 준아가 쓴다. Drive 에서 지우지 않는다.